# Bloom Filters

Bloom Filters are a space-efficient probabilistic data structure used to test whether an element is a member of a set.
    
- They can have false positives but not false negatives.
- This means that if the Bloom Filter says an element is in the set, it might be wrong.
- But if it says an element is not in the set, it is definitely correct.

Working:
- The Bloom Filter uses multiple hash functions to set bits in a bit array.
- When adding an element, the Bloom Filter computes multiple hash values for the element and sets the corresponding bits in the bit array.
- When checking for membership, it computes the same hash values and checks if all corresponding bits are set to 1.
***

# Algorithm
## 1. The Components
To initialize a Bloom Filter, you need:
1.  **A Bit Array ($B$):** An array of $m$ bits, all initially set to $0$.
2.  **Hash Functions ($k$):** A set of $k$ different hash functions $\{h_1, h_2, \dots, h_k\}$.
    * Each hash function must map an input element to one of the $m$ array positions.
    * They should be independent and uniformly distributed.

---
## 2. Operations
### A. Insertion
To add an element $x$ to the filter:
1.  Feed the element $x$ into each of the $k$ hash functions.
2.  Each function produces an index: $i_j = h_j(x) \pmod m$ for $j = 1, \dots, k$.
3.  Set the bits at all resulting positions $i_1, i_2, \dots, i_k$ in the bit array to **1**.

### B. Membership Check (Query)
To check if an element $y$ is in the set:
1.  Feed the element $y$ into the same $k$ hash functions to get the indices $i_j = h_j(y) \pmod m$.
2.  Inspect the bits at these indices in the bit array.
3.  **The Result:**
    * If **any** of the bits at these indices is **0**, the element is **definitely not** in the set.
    * If **all** bits are **1**, the element **might** be in the set (subject to false positive probability).

---
## 3. The Mathematics
The performance of a Bloom Filter depends on the relationship between $m$, $n$, and $k$.
- $m$: Total number of bits.
- $k$: Number of hash functions.
- $n$: Number of elements we wish to insert.

### False Positive Probability ($P$)
After inserting $n$ elements into a filter of size $m$ using $k$ hash functions, the probability that a specific bit is still $0$ is:
$$p_0 = \left(1 - \frac{1}{m}\right)^{kn} \approx e^{-kn/m}$$

The probability that a bit is $1$ is therefore $1 - e^{-kn/m}$. A false positive occurs when all $k$ indices for a non-member element happen to be $1$:
$$P \approx \left(1 - e^{-kn/m}\right)^k$$

### Optimal Number of Hash Functions ($k$)
To minimize the false positive probability for a given $m$ and $n$, the optimal number of hash functions is:
$$k = \frac{m}{n} \ln 2 \approx 0.693 \frac{m}{n}$$

### Required Space ($m$)
If you have a target false positive rate $P$ and a known number of elements $n$, the required number of bits $m$ is:
$$m = -\frac{n \ln P}{(\ln 2)^2}$$

---
## 4. Summary Table

| Feature | Description |
| :--- | :--- |
| **Space Complexity** | $O(m)$ bits, regardless of element size. |
| **Time Complexity** | $O(k)$ for both insertion and lookup. |
| **False Negatives** | Impossible ($0\%$ probability). |
| **False Positives** | Possible; probability decreases as $m/n$ increases. |
| **Deletion** | Not supported in standard Bloom Filters (requires Counting Bloom Filters). |
---

In [1]:
import math as mt
from hashlib import md5, sha256

class BloomFilter:
    def __init__(self, n, p):
        self.n = n # Expected number of elements to be stored in the Bloom Filter
        self.p = p
        self.c = mt.log(2)
        self.c2 = self.c ** 2
        self.t = mt.log(1 / self.p)
        self.m = abs(int(-self.n * self.t / self.c2))  # Size of the bit array
        self.k = max(1, int(self.m / self.n * self.c)) # Ensure at least one hash function is used

        self.bit_array = [0] * self.m

    def hash1(self, item):
        return int(md5(item.encode()).hexdigest(), 16) % self.m

    def hash2(self, item):
        return int(sha256(item.encode()).hexdigest(), 16) % self.m

    def add(self, item):
        item = str(item)
        h1 = self.hash1(item)
        h2 = self.hash2(item)
        for i in range(self.k):
            index = (h1 + i * h2) % self.m
            self.bit_array[index] = 1
    
    def check(self, item):
        for i in range(self.k):
            index = (self.hash1(item) + i * self.hash2(item)) % self.m
            # If any bit at index is 0, the item is definitely not in the set
            if self.bit_array[index] == 0:
                return False
        # If all bits are 1, the item might be in the set (with a certain probability of false positives)
        return True

## Driver function

In [2]:
if __name__ == "__main__":
    bloom = BloomFilter(n = 4, p = 0.001)
    bloom.add("hello")
    bloom.add("world")
    bloom.add("example")

    print(bloom.check("hello"))   # Output: True
    print(bloom.check("world"))   # Output: True
    print(bloom.check("example")) # Output: True
    print(bloom.check("test"))    # Output: False (most likely)

True
True
True
False


***